### Tools


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model = "qwen/qwen3-32b"
)
model


ChatGroq(metadata={'versions': {'langchain-core': '1.4.6', 'langchain': '1.3.7'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x1104da660>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1104db0e0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
from langchain.chat_models import init_chat_model

agent = init_chat_model(
    model = "groq:qwen/qwen3-32b"
)
response = agent.invoke("Richest man in the world?")
response.content

"<think>\nOkay, so I need to figure out who the richest person in the world is right now. Let me start by recalling the names I've heard in the news. Elon Musk comes to mind because of Tesla and SpaceX. Then there's Jeff Bezos from Amazon. Bernard Arnault is the head of LVMH, which is a big luxury goods company. Maybe Mark Zuckerberg from Meta? Also, maybe someone like Bill Gates or Warren Buffett still in the top?\n\nWait, I should check the latest net worths. Net worth can fluctuate a lot based on stock prices and company performance. For example, if Tesla's stock goes up, Elon Musk's net worth increases. Similarly, if Amazon's stock is down, Jeff Bezos' net worth might decrease. So timing is important here.\n\nI remember that in 2021, Elon Musk briefly became the richest person due to the surge in Tesla's stock. But then maybe Jeff Bezos took over again. Now, in 2023, I think Bernard Arnault has been rising in the rankings because of the luxury goods market doing well, especially po

### Tools calling

from langchain_tools import tools --> now this tool can be used as a decorator @tool and any funciton can be used a tool


In [ ]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get weather of the city"""
    return f"The weather of {city} is sunny!"

model_with_tools = model.bind_tools([get_weather])


In [ ]:
response = model_with_tools.invoke("What is the weather like in Boston?")

response


AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a city parameter. Since the user mentioned Boston, I need to call that function with "Boston" as the city argument. I\'ll make sure to format the tool call correctly within the XML tags.\n', 'tool_calls': [{'id': 'e44vgwn02', 'function': {'arguments': '{"city":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 153, 'total_tokens': 245, 'completion_time': 0.126274849, 'completion_tokens_details': {'reasoning_tokens': 68}, 'prompt_time': 0.014494101, 'prompt_tokens_details': None, 'queue_time': 0.409680139, 'total_time': 0.14076895}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, 

### Tool execution loop


In [ ]:
#Step 1: Model Generates tool calls
messages = [{"role": "user", "content" : "What is the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
print("ai_msg: ", ai_msg)
messages.append(ai_msg)
print("messages: ", messages)


#Step 2: Exectue Tools and collect results
for tool_call in ai_msg.tool_calls:
    #Excecute the tool with generated arguments
    print("tool_call:", tool_call)
    tool_result = get_weather.invoke(tool_call)
    print("tool_result: ", tool_result)
    messages.append(tool_result)
    print("messages: ", messages)

final_response = model_with_tools.invoke(messages)
print("final_response: ", final_response)


ai_msg:  content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a city parameter. Since the user specified Boston, I need to call this function with "Boston" as the city argument. I\'ll make sure the JSON is correctly formatted with the city name in the parameters. No other tools are available, so this should be the only function call needed.\n', 'tool_calls': [{'id': 'nf3r5jkk2', 'function': {'arguments': '{"city":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 110, 'prompt_tokens': 152, 'total_tokens': 262, 'completion_time': 0.196195531, 'completion_tokens_details': {'reasoning_tokens': 86}, 'prompt_time': 0.016766696, 'prompt_tokens_details': None, 'queue_time': 0.203316844, 'total_time': 0.212962227}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier'